# 04 — Statistical Analysis & Hypothesis Testing
## Hospital Readmission Risk Prediction System (DISCHARGE)
**Dataset:** UCI Diabetes 130-US Hospitals (1999–2008) · Cleaned Staging Encounters (N = 99,340)  
**Roadmap Tasks:** `P3-07` (Statistical Tests), `P3-08` (Odds Ratios via Multivariable Logistic Regression)  
**Objective:** Statistically validate clinical hypotheses regarding factors associated with 30-day unplanned readmission. Applies Bonferroni multiplicity adjustments, Pearson Chi-square tests of independence, Mann-Whitney U tests, Cramér's V effect sizes, and multivariable logistic regression odds ratios.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

from readmission.config import (
    REPO_ROOT,
    REPORTS_EDA_DIR,
    REPORTS_FIGURES_DIR,
    RANDOM_STATE,
)
from readmission.data.load import load_staging_encounters
from readmission.features.icd9 import add_diagnosis_groups

print('Statistical libraries imported.')

### 1. Load Cleaned Staging Cohort
Pulling full dataset of 99,340 encounters and applying clinical mappings.

In [ ]:
df = load_staging_encounters()
df = add_diagnosis_groups(df)

# Admission and discharge groupings
def map_discharge_group(desc: str) -> str:
    if not desc or pd.isna(desc):
        return 'Other/Unknown'
    d = str(desc).lower()
    if 'home with home health' in d:
        return 'Home Health'
    if 'home' in d and 'iv' not in d:
        return 'Home'
    if 'snf' in d or 'nursing' in d or 'icf' in d or 'rehab' in d:
        return 'Facility (SNF/ICF/Rehab)'
    if 'short term hospital' in d or 'inpatient care' in d:
        return 'Transfer Hospital'
    return 'Other/Unknown'

df['discharge_grp'] = df['discharge_disposition_desc'].map(map_discharge_group)
print(f'Ready for statistical modeling: {len(df):,} encounters.')

### 2. Statistical Hypothesis Test Results (`P3-07`)
Load the computed hypothesis test results (`reports/eda/stats_tests.csv`). Contains 21 hypothesis tests with Bonferroni multiplicity adjustments.

In [ ]:
stats_csv = REPORTS_EDA_DIR / 'stats_tests.csv'
stats_df = pd.read_csv(stats_csv)
print(f'Total hypothesis tests evaluated: {len(stats_df)}')
stats_df[['feature', 'variable_type', 'test_name', 'statistic_value', 'raw_p_value', 'bonferroni_p_value', 'effect_size_name', 'effect_size_value', 'significance_tier']]

### 3. Top Predictors Ranked by Effect Size
Sort categorical features by Cramér's V and numeric features by rank-biserial effect size.

In [ ]:
print('Top Categorical Features by Cramér\'s V:')
display(stats_df[stats_df['variable_type'] == 'categorical'].sort_values('effect_size_value', ascending=False)[['feature', 'statistic_value', 'degrees_of_freedom', 'effect_size_value', 'significance_tier']])

print('\nTop Numeric Features by Rank-Biserial r:')
display(stats_df[stats_df['variable_type'] == 'numeric'].sort_values('effect_size_value', ascending=False)[['feature', 'non_readmitted_summary', 'readmitted_summary', 'effect_size_value', 'significance_tier']])

### 4. Multivariable Logistic Regression & Odds Ratios (`P3-08`)
Fit a multivariable logistic regression model using `statsmodels` to estimate adjusted Odds Ratios (OR) and 95% Confidence Intervals for key clinical factors.

In [ ]:
formula = (
    "readmitted_30d ~ time_in_hospital + number_inpatient + number_emergency + "
    "number_diagnoses + num_medications + "
    "C(a1c_result, Treatment(reference='NotMeasured')) + "
    "C(change, Treatment(reference='No')) + "
    "C(discharge_grp, Treatment(reference='Home')) + "
    "C(diag_1_group, Treatment(reference='Circulatory'))"
)
logit_model = smf.logit(formula, data=df).fit()
print(logit_model.summary().tables[0])

### 5. Adjusted Odds Ratios with 95% Confidence Intervals
Extract exponentiated parameters ($e^\beta$) representing adjusted Odds Ratios.

In [ ]:
params = logit_model.params
conf = logit_model.conf_int()
or_table = pd.DataFrame({
    'Odds Ratio (OR)': np.exp(params).round(3),
    '95% Lower CI': np.exp(conf[0]).round(3),
    '95% Upper CI': np.exp(conf[1]).round(3),
    'p-value': logit_model.pvalues.apply(lambda p: '< 0.001' if p < 0.001 else f'{p:.4f}')
}).drop(index='Intercept')
or_table

### 6. Replication of Strack et al. (2014) Findings
In Strack et al. (2014), the authors demonstrated that when HbA1c is measured and elevated ($>8\%$) with concurrent medication changes, readmission rates are reduced relative to patients whose HbA1c was unmeasured. Here we verify the effect in our cleaned cohort.

In [ ]:
a1c_sub = df.groupby(['a1c_result', 'change'])['readmitted_30d'].agg(['count', 'mean'])
a1c_sub['readmit_rate_%'] = (a1c_sub['mean'] * 100).round(2)
a1c_sub